In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [68]:
df = pd.read_csv('datasets/heart-disease.csv')
df

train = df.sample(frac=0.8, random_state=4242)
test = df.drop(train.index)

y_train = train['target'].to_numpy()
X_train = train.drop(columns='target').to_numpy()
y_test = test['target'].to_numpy()
X_test = test.drop(columns='target').to_numpy()


In [69]:
def euclidian_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

cov = np.cov(X_train.T)
cov += 1e-6 * np.eye(cov.shape[0])
cov_inv = np.linalg.inv(cov)

def mahalanobis_distance(x1, x2):
    diff = x1 - x2
    return np.sqrt(diff.T @ cov_inv @ diff)

k = 0
k_best = None
best_accuracy = 0


for k in range(1, 10):
    predictions = []
    for x in X_test:
        distances = []
        for i, x_train in enumerate(X_train):
            # distances.append(( euclidian_distance(x, x_train), y_train[i] ))
            distances.append(( mahalanobis_distance(x, x_train), y_train[i] ))

        distances.sort(key=lambda x : x[0])
        k_neighbours = distances[:k]
        # labels = [label for dist, label in k_neighbours]
        # prediction = max(set(labels), key=labels.count)

        counts = {}
        max_count = 0
        prediction = None
        for dist, label in k_neighbours:
            w = 1 / (dist + 1e-6)
            if label not in counts:
                counts[label] = 0
            counts[label] += w
            if counts[label] > max_count:
                max_count = counts[label]
                prediction = label
            
        predictions.append(prediction)

    predictions = np.array(predictions)
    accuracy = np.sum(predictions == y_test) / len(y_test) * 100
    if(accuracy > best_accuracy):
        best_accuracy = accuracy
        k_best = k






# def euclidean_distance(x1, x2):
#     return np.sqrt(np.sum((x1 - x2) ** 2))


# # Covariance matrix
# cov = np.cov(X_train, rowvar=False)

# # Regularization
# cov += 1e-6 * np.eye(cov.shape[0])

# # Inverse covariance
# cov_inv = np.linalg.inv(cov)


# def mahalanobis_distance(x1, x2):
#     diff = x1 - x2
#     return np.sqrt(diff.T @ cov_inv @ diff)


# k_best = None
# best_accuracy = 0

# # Ensure labels are 1D
# y_train = np.asarray(y_train).ravel()
# y_test = np.asarray(y_test).ravel()

# for k in range(1, 10):

#     predictions = []

#     for x in X_test:

#         distances = []

#         for i, x_train in enumerate(X_train):
#             d = mahalanobis_distance(x, x_train)
#             distances.append((d, y_train[i]))

#         distances.sort(key=lambda x: x[0])

#         k_neighbours = distances[:k]

#         # Distance-weighted voting
#         counts = {}

#         for dist, label in k_neighbours:

#             weight = 1 / (dist + 1e-6)

#             if label not in counts:
#                 counts[label] = 0

#             counts[label] += weight

#         # Find class with maximum weight
#         prediction = None
#         max_weight = -1

#         for label, weight in counts.items():

#             if weight > max_weight:
#                 max_weight = weight
#                 prediction = label

#         predictions.append(prediction)

#     predictions = np.array(predictions)

#     accuracy = np.mean(predictions == y_test) * 100

#     print(f"k = {k}, Accuracy = {accuracy:.2f}%")

#     if accuracy > best_accuracy:
#         best_accuracy = accuracy
#         k_best = k

# print("\nBest k:", k_best)
# print("Best Accuracy:", best_accuracy)


In [70]:
print(f'Best k: {k_best}, Best Accuracy: {best_accuracy}')

Best k: 9, Best Accuracy: 83.60655737704919
